## Démarrage de la session Spark

In [1]:
from pyspark.sql import SparkSession

# Création de la session Spark
spark = SparkSession.builder \
    .appName("Projet9_Extraction_Fruits") \
    .getOrCreate()

# Vérification que Spark tourne bien
print("Session Spark démarrée avec succès !")
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1770775818407_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Session Spark d?marr?e avec succ?s !

## Définition des PATH pour charger les images et enregistrer les résultats

In [4]:
# Configuration des accès
PATH = 's3://p9-data-mathias-f'
PATH_Data = PATH + '/Test'
PATH_Result = PATH + '/Results'

print('PATH:        ' + PATH)
print('PATH_Data:   ' + PATH_Data)
print('PATH_Result: ' + PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

PATH:        s3://p9-data-mathias-f
PATH_Data:   s3://p9-data-mathias-f/Test
PATH_Result: s3://p9-data-mathias-f/Results

In [6]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
images.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+------+--------------------+
|                path|   modificationTime|length|             content|
+--------------------+-------------------+------+--------------------+
|s3://p9-data-math...|2026-02-11 01:09:03|  7353|[FF D8 FF E0 00 1...|
|s3://p9-data-math...|2026-02-11 01:09:03|  7350|[FF D8 FF E0 00 1...|
|s3://p9-data-math...|2026-02-11 01:09:03|  7349|[FF D8 FF E0 00 1...|
|s3://p9-data-math...|2026-02-11 01:09:03|  7348|[FF D8 FF E0 00 1...|
|s3://p9-data-math...|2026-02-11 01:09:03|  7328|[FF D8 FF E0 00 1...|
+--------------------+-------------------+------+--------------------+
only showing top 5 rows

In [9]:
from pyspark.sql.functions import split, element_at

# Extraction du label (le nom du dossier parent de l'image)
images = images.withColumn('label', element_at(split(images['path'], '/'), -2))

# Affichage de la structure pour vérifier la nouvelle colonne 'label'
images.printSchema()

# Affichage des 5 premières lignes
images.select('path', 'label').show(5, False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

+----------------------------------------------------+----------+
|path                                                |label     |
+----------------------------------------------------+----------+
|s3://p9-data-mathias-f/Test/Watermelon/r_106_100.jpg|Watermelon|
|s3://p9-data-mathias-f/Test/Watermelon/r_109_100.jpg|Watermelon|
|s3://p9-data-mathias-f/Test/Watermelon/r_108_100.jpg|Watermelon|
|s3://p9-data-mathias-f/Test/Watermelon/r_107_100.jpg|Watermelon|
|s3://p9-data-mathias-f/Test/Watermelon/r_95_100.jpg |Watermelon|
+----------------------------------------------------+----------+
only showing top 5 rows

## Préparation du modèle

In [11]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array

# Chargement du modèle MobileNetV2 pré-entraîné sur ImageNet
model = MobileNetV2(weights='imagenet', 
                    include_top=True, 
                    input_shape=(224, 224, 3))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

14536120/14536120 [==============================] - 1s 0us/step

In [14]:
from tensorflow.keras.models import Model

# On crée le nouveau modèle en prenant l'entrée de MobileNetV2 
# et en s'arrêtant à l'avant-dernière couche (layers[-2])
new_model = Model(inputs=model.input, 
                  outputs=model.layers[-2].output)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
broadcast_weights = sc.broadcast(new_model.get_weights())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [24]:
new_model.summary()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 112, 112, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 112, 112, 32  128         ['Conv1[0][0]']                  
                                )                                                           

In [32]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(brodcast_weights.value)
    return new_model

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Définition du processus de chargement des images et application de leur featurisation à travers l'utilisation de pandas UDF

In [33]:
from pyspark.sql.functions import pandas_udf, PandasUDFType, col, split, element_at

def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Exécutions des actions d'extractions de features

In [20]:
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [34]:
# --- ÉTAPE 1 : IMPORTS (Indispensables pour les Workers) ---
import pandas as pd
import numpy as np
import io
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Model
from pyspark.sql.functions import pandas_udf, PandasUDFType, col, split, element_at

# --- ÉTAPE 2 : MODÈLE ET BROADCAST ---
# On prépare le modèle sur le Master
model_base = MobileNetV2(weights='imagenet', include_top=True, input_shape=(224, 224, 3))
for layer in model_base.layers:
    layer.trainable = False
model_extracteur = Model(inputs=model_base.input, outputs=model_base.layers[-2].output)

# On diffuse les poids (Vérifiez bien l'orthographe : broadcast_weights)
broadcast_weights = spark.sparkContext.broadcast(model_extracteur.get_weights())

# --- ÉTAPE 3 : FONCTIONS DE CALCUL ---
def model_fn():
    model = MobileNetV2(weights='imagenet', include_top=True, input_shape=(224, 224, 3))
    new_model = Model(inputs=model.input, outputs=model.layers[-2].output)
    new_model.set_weights(broadcast_weights.value)
    return new_model

def preprocess(content):
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    # C'est ici que 'np' posait problème !
    input_data = np.stack(content_series.map(preprocess))
    preds = model.predict(input_data)
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

# --- ÉTAPE 4 : ACTION ---
# On s'assure que 'images' contient bien la colonne 'content'
# Puis on lance l'extraction
print("Calcul en cours sur le cluster...")

# On extrait le label au passage
images_final = images.withColumn('label', element_at(split(col('path'), '/'), -2))

features_df = images_final.repartition(24).select(
    col("path"),
    col("label"),
    featurize_udf("content").alias("features")
)

# Cette fois, show(5) devrait fonctionner !
features_df.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Calcul en cours sur le cluster...
+--------------------+--------------+--------------------+
|                path|         label|            features|
+--------------------+--------------+--------------------+
|s3://p9-data-math...|    Watermelon|[0.24856485, 0.37...|
|s3://p9-data-math...|    Watermelon|[0.5976525, 0.095...|
|s3://p9-data-math...|    Watermelon|[0.5296135, 0.097...|
|s3://p9-data-math...|Pineapple Mini|[0.0, 4.478077, 0...|
|s3://p9-data-math...|Pineapple Mini|[0.0, 4.621436, 0...|
+--------------------+--------------+--------------------+
only showing top 5 rows

In [35]:
print(PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3://p9-data-mathias-f/Results

In [36]:
features_df.write.mode("overwrite").parquet(PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Chargement des données enregistrées et validation du résultat

In [38]:
# On lit le fichier Parquet avec Spark au lieu de Pandas
df_spark = spark.read.parquet(PATH_Result)

# On affiche les premières lignes
df_spark.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+--------------+--------------------+
|                path|         label|            features|
+--------------------+--------------+--------------------+
|s3://p9-data-math...|    Watermelon|[1.0782965, 0.563...|
|s3://p9-data-math...|    Watermelon|[0.0016325737, 0....|
|s3://p9-data-math...|    Watermelon|[0.13241422, 0.22...|
|s3://p9-data-math...|Pineapple Mini|[0.0, 4.5370426, ...|
|s3://p9-data-math...|    Watermelon|[0.0, 0.91131, 0....|
+--------------------+--------------+--------------------+
only showing top 5 rows

In [39]:
# Pour obtenir le nombre de lignes et de colonnes (df.shape)
print((df_spark.count(), len(df_spark.columns)))

# Pour voir la taille du vecteur de features de la première ligne
first_row = df_spark.select("features").first()
print(f"Dimension des features : {len(first_row['features'])}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

(22688, 3)
Dimension des features : 1280

## PCA 

In [40]:
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf, col

# --- ÉTAPE OPTIONNELLE : Récupération des données ---
try:
    features_df.printSchema()
except NameError:
    print("Chargement des données depuis S3...")
    features_df = spark.read.parquet(PATH_Result)

# --- DÉBUT DU TRAITEMENT PCA ---

# 1. Conversion Array<Float> -> Vector (Dense)
# On s'assure que Spark ML peut lire les données
to_vector_udf = udf(lambda a: Vectors.dense(a), VectorUDT())
features_df_vector = features_df.withColumn("features_vec", to_vector_udf("features"))

# 2. Configuration de la PCA
# ATTENTION : k doit être <= au nombre d'images total
n_images = features_df.count()
k_pca = 50 if n_images > 50 else n_images
print(f"Nombre d'images : {n_images}, PCA configurée sur k={k_pca}")

pca = PCA(k=k_pca, inputCol="features_vec", outputCol="pca_features")

# 3. Entraînement du modèle
# Spark calcule les composantes principales
model_pca = pca.fit(features_df_vector)

# 4. Transformation
result_pca = model_pca.transform(features_df_vector)

# 5. Sélection et sauvegarde
final_df = result_pca.select("path", "label", "pca_features")

# Affichage pour vérification
final_df.show(5, truncate=True)

# --- SAUVEGARDE ---
PATH_Result_PCA = PATH_Result + "_PCA"
print(f"Sauvegarde en cours dans : {PATH_Result_PCA}")

# On utilise 'overwrite' pour pouvoir relancer le code plusieurs fois sans erreur
final_df.write.mode("overwrite").parquet(PATH_Result_PCA)
print("✅ PCA terminée et sauvegardée !")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- path: string (nullable = true)
 |-- label: string (nullable = true)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = true)

Nombre d'images : 22688, PCA configur?e sur k=50
+--------------------+--------------+--------------------+
|                path|         label|        pca_features|
+--------------------+--------------+--------------------+
|s3://p9-data-math...|    Watermelon|[-3.6636958702155...|
|s3://p9-data-math...|    Watermelon|[-3.8404746781603...|
|s3://p9-data-math...|    Watermelon|[-4.6430028773897...|
|s3://p9-data-math...|Pineapple Mini|[-5.3932783395997...|
|s3://p9-data-math...|Pineapple Mini|[-5.1488994675521...|
+--------------------+--------------+--------------------+
only showing top 5 rows

Sauvegarde en cours dans : s3://p9-data-mathias-f/Results_PCA
? PCA termin?e et sauvegard?e !